In [5]:
import sys
import re
import string
from collections import Counter
import nltk

In [2]:
decompressed_bytes = base64.b64decode(embedded_string)
corpus_text_n = zlib.decompress(decompressed_bytes).decode('utf-8')
# 3. Turn it into a vocabulary list safely without NLTK's punkt downloader
corpus_words = corpus_text_n.split()

In [18]:
eng_words = set()
vocab = Counter()
nltk_text = ""

eng_words = set(w for w in corpus_words)
nltk_text += " ".join(corpus_words)
vocab.update([w for w in corpus_words])


input_path = '/home/david/Documents/HackerRank-challenges/the-missing-characters/the-missing-characters-testcases/input/input01.txt'

In [19]:
# --- Clean and normalize texts ---
def clean_text(text):
    # Discard punctuation to allow robust n-gram string matching
    t = re.sub(r'[^a-z\s#]', '', text.lower())
    return re.sub(r'\s+', ' ', t)

# Dictionary mapping standard English letter frequencies (for tie-breaking edge cases)
letter_freq = {
    'e': 13.0, 't': 9.1, 'a': 8.2, 'o': 7.5, 'i': 7.0, 'n': 6.7, 's': 6.3, 'h': 6.1, 'r': 6.0, 
    'd': 4.3, 'l': 4.0, 'c': 2.8, 'u': 2.8, 'm': 2.5, 'w': 2.4, 'f': 2.2, 'g': 2.0, 'y': 2.0, 
    'p': 1.9, 'b': 1.5, 'v': 0.98, 'k': 0.77, 'j': 0.15, 'x': 0.15, 'q': 0.09, 'z': 0.07
}

with open(input_path, 'r', encoding='utf-8') as f:
    content = f.read().strip()
    cleaned_input = clean_text(content)

cleaned_nltk = clean_text(nltk_text)

def find_best_char(candidates, context_pairs):
    """Finds the optimal missing character by string-matching contexts of decreasing size."""
    for lp, rp in context_pairs:
        current_scores = {}
        for c in candidates:
            # Reconstruct string sequence replacing '#' with the candidate letter (case agnostic)
            query = lp + c + rp
            score = cleaned_nltk.count(query)
                
            if score > 0:
               current_scores[c] = score
                    
        if current_scores:
            # Return character with the highest n-gram match score (break ties using standard frequencies)
            return max(current_scores) # max(current_scores, key=lambda x: (current_scores[x], letter_freq.get(x, 0)))
    return None

In [20]:
# --- Process each missing character '#' ---
# Find the positions of all '#' tags
hashes = [i for i, char in enumerate(cleaned_input) if char == '#']
missing_characters = []

for h_idx in hashes:
    # a. Isolate the target word containing the '#' symbol
    start = h_idx
    while start > 0 and cleaned_input[start - 1] != ' ':
        start -= 1
    end = h_idx
    while end < len(cleaned_input) - 1 and cleaned_input[end + 1] != ' ':
        end += 1

    word_pattern = cleaned_input[start:end + 1]
    local_idx = h_idx - start

    # b. Find valid substitutions purely using dictionary lookups
    valid_candidates = []
    if '#' not in (word_pattern[:local_idx] + word_pattern[local_idx + 1:]):
        for c in string.ascii_lowercase:
            candidate_word = word_pattern[:local_idx] + c + word_pattern[local_idx + 1:]
            if candidate_word in vocab or candidate_word in eng_words:
                valid_candidates.append(c)

    candidates_to_check = valid_candidates if len(valid_candidates) > 0 else list(string.ascii_lowercase)

    # c. Extract contiguous left and right contexts (safely stopping if another '#' is adjacent)
    left_limit = h_idx - 1
    while left_limit >= 0 and cleaned_input[left_limit] != '#':
        left_limit -= 1
    left_context_full = cleaned_input[left_limit + 1:h_idx]

    right_limit = h_idx + 1
    while right_limit < len(cleaned_input) and cleaned_input[right_limit] != '#':
        right_limit += 1
    right_context_full = cleaned_input[h_idx + 1:right_limit]

    # d. Build context pairs of shrinking sizes
    context_pairs = []
    max_len = max(len(left_context_full), len(right_context_full))
    for w in range(min(40, max_len), 0, -1):
        lp = left_context_full[-w:] if w > 0 else ""
        rp = right_context_full[:w] if w > 0 else ""
        pair = (lp, rp)
        if not context_pairs or context_pairs[-1] != pair:
            context_pairs.append(pair)

    # e. Evaluate candidates
    best_char = None

    if len(valid_candidates) == 1:
        best_char = valid_candidates[0]
    else:
        best_char = find_best_char(candidates_to_check, context_pairs)

    if not best_char:
        unigram_scores = {}
        for c in candidates_to_check:
            unigram_scores[c] = cleaned_nltk.count(c)

        if unigram_scores:
            best_char = max(unigram_scores, key=lambda x: (unigram_scores[x], letter_freq.get(x, 0)))
        else:
            best_char = 'e'

    print(best_char)

y
k
r
l
k
n
i
n
r
t
s
i
a
n


In [21]:
output_path = '/home/david/Documents/HackerRank-challenges/the-missing-characters/the-missing-characters-testcases/output/output01.txt'

with open(output_path, 'r', encoding='utf-8') as f:
    output = f.read().strip()

output = output.split('\n')

In [22]:
missing = []
for a, b in zip(missing_characters, output):
    missing.append(a != b)

In [23]:
sum(missing)

0

In [1]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

# Make sure you have the tokenizers downloaded
nltk.download('punkt', quiet=True)

file_path = '/home/david/Documents/HackerRank-challenges/the-missing-characters/cleaned_corpus.txt'

# 1. Read the file safely using Python's built-in reader 
# (This completely bypasses the buggy NLTK chunk reader)
with open(file_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()

# 2. Use NLTK to break the raw text into a list of sentences
sentences = sent_tokenize(raw_text)

# 3. Replicate the PlaintextCorpusReader output (a list of tokenized word lists)
corpus_sents = [word_tokenize(sent) for sent in sentences]
corpus_words = word_tokenize(raw_text)

# 4. Test it!
print("First sentence:")
print(corpus_sents[0])

print("\nTotal words in corpus:", len(corpus_words))
print("Total sentences in corpus:", len(corpus_sents))

First sentence:
['\ufeffProject', 'Gutenberg', "'s", 'The', 'Secret', 'Cache', ',', 'by', 'E.', 'C.', '[', 'Ethel', 'Claire', ']', 'Brill', 'This', 'eBook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'at', 'no', 'cost', 'and', 'with', 'almost', 'no', 'restrictions', 'whatsoever', '.']

Total words in corpus: 685452
Total sentences in corpus: 29131


In [2]:
# Open the original file in binary read mode
with open('corpus.txt', 'rb') as file:
    raw_content = file.read()

# Replace any Windows (\r\n) or old Mac (\r) line endings with standard Unix (\n)
clean_content = raw_content.replace(b'\r\n', b'\n').replace(b'\r', b'\n')

# Write the clean content back to your cleaned text file
with open('cleaned_corpus.txt', 'wb') as file:
    file.write(clean_content)

In [1]:
import zlib
import base64
import re

# 1. Read your successfully loaded text file
file_path = 'cleaned_corpus.txt'
with open(file_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()

# 2. Aggressively clean the text to save space
# We only need lowercase letters and spaces for the missing character challenge
clean_text = re.sub(r'[^a-z\s]', '', raw_text.lower())
clean_text = re.sub(r'\s+', ' ', clean_text).strip()

# 3. Compress and encode the text
compressed_bytes = zlib.compress(clean_text.encode('utf-8'))
embedded_string = base64.b64encode(compressed_bytes).decode('utf-8')

print("--- COPY THE LINE BELOW INTO HACKERRANK ---")
print(f"EMBEDDED_CORPUS = '{embedded_string}'")

--- COPY THE LINE BELOW INTO HACKERRANK ---
EMBEDDED_CORPUS = '...'


In [3]:
decompressed_bytes = base64.b64decode(embedded_string)
corpus_text_n = zlib.decompress(decompressed_bytes).decode('utf-8')
# 3. Turn it into a vocabulary list safely without NLTK's punkt downloader
corpus_words = corpus_text_n.split()